# JEE Main percentile prediction: walkthrough

A guided tour of the full pipeline, from raw tables through Stage 1 fingerprints, exploratory analysis and the cutoff ensemble to honest evaluation. Every number is recomputed live from `data/`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from jee_percentile import plots  # applies the project's chart style
from jee_percentile.data import load_curves, load_dataset
from jee_percentile.curves import fit_all_shifts
pd.set_option('display.precision', 2)

## 1 · The raw material
One row per (shift, percentile tier) from the published tables.

In [2]:
curves = load_curves()
print(f"{curves.shift_id.nunique()} shifts, {len(curves)} published points")
curves.groupby(['year', 'session']).agg(shifts=('shift_id', 'nunique'), tiers=('percentile', 'size'))

41 shifts, 583 published points


shifts  tiers
year session               
2023 January      12     96
2025 April         9    117
     January      10    140
2026 January      10    230

## 2 · Stage 1: two-number fingerprints
Each shift's table is fit with `100·Φ((marks−μ)/σ)`. Low RMSE means the parametric form is a faithful compression of the table.

In [3]:
fits = fit_all_shifts(curves)
fits.groupby('year')[['mu', 'sigma', 'rmse_pct']].describe().loc[:, (slice(None), ['mean', 'max'])]

mu          sigma         rmse_pct      
       mean    max    mean     max     mean   max
year                                             
2023 -51.62 -32.14  101.84  108.79     0.19  0.30
2025  -3.92  21.34   78.87   96.39     0.14  0.34
2026 -19.05  -7.47   76.73   82.24     0.10  0.17

## 3 · Modelling table
Shifts with *both* a curve and difficulty labels; 90/91 tiers imputed where missing.

In [4]:
df = load_dataset()
df[['shift_id', 'phy_hard', 'phy_easy', 'overall_hard', 'is_2026', 'n_imputed', 'score_90.0', 'score_95.0', 'score_99.0']].head(8)

,shift_id,phy_hard,phy_easy,overall_hard,is_2026,n_imputed,score_90.0,score_95.0,score_99.0
0,2025_02_APR_1,0.0,36.0,6.67,0,2,90.45,117.0,173.0
1,2025_02_APR_2,0.0,56.0,2.67,0,2,115.52,147.0,203.0
2,2025_03_APR_1,4.0,52.0,5.33,0,2,102.85,127.0,174.0
3,2025_03_APR_2,0.0,56.0,2.67,0,2,110.16,142.0,201.0
4,2025_04_APR_1,0.0,44.0,2.67,0,2,100.22,126.0,175.0
5,2025_04_APR_2,0.0,44.0,8.00,0,2,97.53,125.0,177.0
6,2025_07_APR_1,0.0,36.0,2.67,0,2,101.63,128.0,177.0
7,2025_07_APR_2,0.0,60.0,1.33,0,2,107.16,134.0,183.0


## 4 · Which difficulty signal matters?

In [5]:
from jee_percentile.evaluate import spearman_table
spearman_table(df).pivot(index='feature', columns='target', values='rho').sort_values('marks@95')

target,marks@90,marks@95,marks@99
feature,,,
overall_hard,-0.68,-0.71,-0.67
phy_hard,-0.52,-0.55,-0.59
math_hard,-0.52,-0.50,-0.40
chem_hard,-0.24,-0.32,-0.34
math_easy,0.06,0.02,0.04
chem_easy,0.04,0.13,0.18
is_evening,0.35,0.38,0.38
overall_easy,0.58,0.63,0.57
phy_easy,0.63,0.65,0.52


## 5 · Leave-one-shift-out evaluation
Every shift is predicted by a model that never saw it.

In [6]:
from jee_percentile.evaluate import run
results = run(write=False)
pd.DataFrame(results['leaderboard'] + results['ablation']).drop_duplicates('name')[['name', 'pct_mae', 'pct_mae_ci', 'marks_mae']].sort_values('pct_mae')

,name,pct_mae,pct_mae_ci,marks_mae
6,Physics + session time,0.70,"(0.49959652132795546, 0.907857196021121)",6.03
2,Cutoff ensemble (deployed),0.72,"(0.5153347720201669, 0.9312567328097274)",6.27
5,Physics difficulty (deployed),0.72,"(0.5153347720201669, 0.9312567328097274)",6.27
7,All three subjects,0.75,"(0.5475529020777692, 0.9709296675983657)",6.68
4,Overall difficulty,0.79,"(0.5748222384820391, 1.0355922992760407)",6.99
1,Year-mean curve,0.84,"(0.6066798403803896, 1.1036330717925689)",7.56
3,Year only,0.89,"(0.6556233684350836, 1.152669027167994)",7.96
0,Mean curve (no features),1.32,"(1.0658850893153449, 1.5742533917190131)",11.82


In [7]:
pd.DataFrame(results['improvement'])

,baseline,mean,ci95,relative,shifts_improved
0,Mean curve (no features),0.61,"(0.4046306160369274, 0.7980817115557243)",0.46,24
1,Year-mean curve,0.12,"(-0.05188540988247408, 0.28848238534982407)",0.15,20


In [8]:
results['prediction_interval']

{'nominal_coverage': 0.8,
 'empirical_coverage': 0.8,
 'mean_width_marks': 25.389755172413793,
 'mean_width_percentile_points': 2.7624421144210807}

## 6 · Using the model

In [9]:
from jee_percentile.train import train
model = train()
for label, hard, easy in [('easy paper', 0, 56), ('typical paper', 4, 40), ('hard paper', 12, 28)]:
    low, pct, high = model.predict_interval(130, model.query(hard, easy))
    print(f'130 marks on a {label:13s} -> {pct:5.2f} percentile  (80% band {low:.2f} - {high:.2f})')

130 marks on a easy paper    -> 96.10 percentile  (80% band 94.49 - 97.37)
130 marks on a typical paper -> 97.07 percentile  (80% band 95.74 - 98.10)
130 marks on a hard paper    -> 97.98 percentile  (80% band 96.89 - 98.57)


In [10]:
import numpy as np
from jee_percentile.config import PERCENTILES
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, hard, easy in [('easy', 0, 56), ('typical', 4, 40), ('hard', 12, 28)]:
    ax.plot(model.predict_curve(model.query(hard, easy))[0], PERCENTILES, marker='o', lw=2.2, label=f'{label} paper')
ax.set(xlabel='Marks needed', ylabel='Percentile', title='Predicted cutoff curves')
ax.legend();

## Takeaways
- Physics difficulty is the most reliable single driver of where percentile cutoffs land.
- A 10-model ridge ensemble plus isotonic regression cuts out-of-sample error by 46% against a difficulty-blind curve.
- The 80% band is calibrated (80% back-tested coverage), so every estimate comes with an honest range.
- The binding constraint is data volume (29 shifts), not model capacity. See the roadmap in the README.